# 1. Instalación de dependencias con resolución de conflictos
# Forzamos la actualización de LangChain y sus integraciones para evitar errores de compatibilidad

In [ ]:

!pip install -q -U langchain langchain-community langchain-experimental langchain-mistralai pandas faiss-cpu tiktoken requests==2.32.4

import requests
import langchain_core
import langchain_experimental
print(f"Instalación completada.")
print(f"Versión de requests: {requests.__version__}")
print(f"Versión de langchain-core: {langchain_core.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
Instalación completada.
Versión de requests: 2.32.4
Versión de langchain-core: 0.3.86


### 2. Normalización y Limpieza de Datos
Antes de pasar los datos a los agentes de IA, debemos asegurarnos de que estén limpios y en un formato estándar.

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Carga con codificación correcta
df_raw = pd.read_csv('sales_data_sample.csv', encoding='latin1')

# 2. Limpieza básica: nombres de columnas en minúsculas y sin espacios
df_raw.columns = df_raw.columns.str.strip().str.lower()

# 3. Conversión de tipos
df_raw['orderdate'] = pd.to_datetime(df_raw['orderdate'])

# 4. Manejo de nulos (ejemplo: rellenar estados vacíos)
df_raw['state'] = df_raw['state'].fillna('N/A')

# 5. Normalización de la columna 'sales' (Escalado entre 0 y 1)
scaler = MinMaxScaler()
df_raw['sales_normalized'] = scaler.fit_transform(df_raw[['sales']])

# Guardamos el dataframe limpio para los siguientes pasos
df = df_raw.copy()

print("Datos normalizados y listos.")
display(df[['orderdate', 'customername', 'sales', 'sales_normalized']].head())

Datos normalizados y listos.


,orderdate,customername,sales,sales_normalized
0,2003-02-24,Land of Toys Inc.,2871.00,0.175644
1,2003-05-07,Reims Collectables,2765.90,0.167916
2,2003-07-01,Lyon Souveniers,3884.34,0.250150
3,2003-08-25,Toys4GrownUps.com,3746.70,0.240030
4,2003-10-10,Corporate Gift Ideas Co.,5205.27,0.347273


```markdown
### Opción 1: Agente de Pandas (Análisis Numérico y Cálculos)
Esta opción es ideal si quieres preguntar cosas como: "¿Cuál fue el promedio de ventas por región?" o "Muéstrame los 5 productos más vendidos".
```

In [ ]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

# Creamos el agente usando el 'llm' definido arriba y el 'df' normalizado
try:
    agente_pandas = create_pandas_dataframe_agent(
        llm,
        df,
        verbose=True,
        allow_dangerous_code=True
    )
    print("Agente de Pandas configurado correctamente con los datos normalizados.")
except Exception as e:
    print(f"Error al configurar el agente: {e}")

Agente de Pandas configurado correctamente con los datos normalizados.


In [ ]:
# 2. Configuración de Mistral AI
import pandas as pd
from google.colab import userdata
from langchain_mistralai.chat_models import ChatMistralAI

try:
    # Obtener API Key de Secrets
    mi_llave_mistral = userdata.get('API_KEY_MISTRAL')

    # Inicializar el modelo
    llm = ChatMistralAI(
        mistral_api_key=mi_llave_mistral,
        model="open-mixtral-8x7b",
        temperature=0
    )

    # Prueba rápida
    respuesta = llm.invoke("Hola Mistral. Confirma que estás listo para analizar datos en una frase.")
    print("Conexión exitosa con Mistral:")
    print(respuesta.content)
except Exception as e:
    print(f"Error al configurar Mistral: {e}. Asegúrate de tener el secreto 'API_KEY_MISTRAL' configurado.")

Conexión exitosa con Mistral:
¡Listo para analizar datos con precisión y eficiencia! 🚀


```markdown
### Opción 2: RAG Tradicional (Búsqueda Semántica)
Esta opción es mejor si el CSV contiene descripciones, comentarios o si buscas políticas financieras basadas en texto.
```

In [ ]:
from langchain_mistralai.embeddings import MistralAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA

# 1. Cargar el CSV usando la codificación detectada
loader = CSVLoader(file_path='sales_data_sample.csv', encoding='latin1')
documentos = loader.load()

# 2. Fragmentar el texto para el procesamiento de la IA
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs_fragmentados = splitter.split_documents(documentos)

# 3. Inicializar Embeddings y Vector Store
try:
    embeddings = MistralAIEmbeddings(mistral_api_key=mi_llave_mistral)
    vectorstore = FAISS.from_documents(docs_fragmentados, embeddings)

    # 4. Crear la cadena de consulta (RetrievalQA)
    rag_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectorstore.as_retriever()
    )
    print("Flujo RAG configurado y listo para búsquedas semánticas.")
except Exception as e:
    print(f"Error al configurar RAG: {e}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

Flujo RAG configurado y listo para búsquedas semánticas.


In [ ]:
def chatbot_mistral_rag():
    print("--- 🤖 Asistente de Ventas Mistral (Modo RAG Activo) ---")
    print("Escribe 'salir' para finalizar.")

    while True:
        pregunta = input("\nTu pregunta para Mistral: ")
        if pregunta.lower() in ['salir', 'exit', 'quit']:
            print("¡Sesión finalizada!")
            break

        try:
            print("[Mistral procesando búsqueda semántica...]")
            # Usamos exclusivamente la cadena RAG con Mistral
            respuesta = rag_chain.invoke(pregunta)
            print(f"\nRespuesta de Mistral AI:\n{respuesta['result']}")
        except Exception as e:
            print(f"Error en la consulta RAG: {e}")

# Iniciar el bot especializado en RAG
chatbot_mistral_rag()

--- 🤖 Asistente de Ventas Mistral (Modo RAG Activo) ---
Escribe 'salir' para finalizar.

Tu pregunta para Mistral: Hola
[Mistral procesando búsqueda semántica...]

Respuesta de Mistral AI:
¡Hola! ¿En qué puedo ayudarte hoy?

Tu pregunta para Mistral: quiero saber cuantas ciudades encuentras en mi dataset
[Mistral procesando búsqueda semántica...]

Respuesta de Mistral AI:
En el dataset proporcionado, se mencionan las siguientes ciudades:

1. Madrid
2. Barcelona

Por lo tanto, hay **2 ciudades** en el dataset.

Tu pregunta para Mistral: salir
¡Sesión finalizada!
